# 04 — Task 4: Reinforcement Learning with Human Feedback (RLHF)

**Project:** Unsupervised Neural Network for Multi-Genre Music Generation  
**Course:** CSE425/EEE474 — Neural Networks  
**Author:** [Your Name]  
**Date:** Spring 2026  

---

This notebook applies **Reinforcement Learning with Human Feedback (RLHF)** to
fine-tune the pretrained Transformer music-generation model from Task 3.

### Mathematical Framework

| Symbol | Meaning |
|--------|---------|
| $X_{\text{gen}} \sim p_\theta(X)$ | Generated music sequence sampled from the policy |
| $r = \text{HumanScore}(X_{\text{gen}})$ | Reward signal (simulated via automated metrics) |
| $\max_\theta \; \mathbb{E}[r(X_{\text{gen}})]$ | Objective: maximize expected reward |
| $\nabla_\theta J(\theta) = \mathbb{E}[r \cdot \nabla_\theta \log p_\theta(X)]$ | REINFORCE policy gradient |

### Notebook Outline

1. **Setup** — imports, model definition, pretrained weights  
2. **Reward Function Design** — automated proxy for human preferences  
3. **RLHF Training** — REINFORCE policy-gradient fine-tuning  
4. **Visualizations** — learning curves, reward distributions, piano rolls  
5. **Human Listening Survey Simulation** — simulated MOS-style scores  
6. **Final Generated Outputs** — MIDI compositions & model checkpoint  
7. **Before vs After Analysis** — radar chart & metrics comparison

---
## Section 1 — Setup & Model Re-implementation

### NVIDIA GPU (why training shows `cpu`)

A plain `pip install torch` from PyPI is usually **CPU-only** — then `torch.version.cuda` is `None` and `torch.cuda.is_available()` is `False`. Your RTX 4090 is fine; PyTorch simply was not installed with CUDA libraries.

**Fix:** run the **next code cell once** with `INSTALL_CUDA_PYTORCH = True`, then **Restart Kernel**, set it back to `False`, and **Run All** from the top.

RTX 40-series works well with **CUDA 12.x** wheels (`cu124`).

---

**If training crashes with `RuntimeError: CUDA error: unknown error`** (often during `MSELoss`): on Windows, **cuDNN is disabled by default** in the setup cell below so LSTM uses stable native GPU kernels. To try faster cuDNN again, set environment variable `PYTORCH_DISABLE_CUDNN=0` before launching Jupyter/Cursor.

In [ ]:
"""GPU bootstrap — run before training if `torch.version.cuda` is None."""

import shutil
import subprocess
import sys

# -----------------------------------------------------------------------------
# PyTorch from PyPI is CPU-only unless you install CUDA wheels from pytorch.org.
# Set True ONCE → run this cell → Restart Kernel → set False → Run All.
# -----------------------------------------------------------------------------
INSTALL_CUDA_PYTORCH = False


def _nvidia_smi_list():
    """Return human-readable GPU list from nvidia-smi, or an error string."""
    exe = shutil.which("nvidia-smi")
    if not exe:
        return False, "nvidia-smi not found on PATH — install NVIDIA drivers from https://www.nvidia.com/drivers/"
    try:
        proc = subprocess.run(
            [exe, "-L"],
            capture_output=True,
            text=True,
            timeout=20,
            check=False,
        )
        return True, proc.stdout.strip() or proc.stderr.strip()
    except Exception as exc:
        return False, str(exc)


def main():
    import torch

    cuda_compiled = torch.version.cuda  # None → CPU-only wheel
    cuda_ok = torch.cuda.is_available()

    print(f"torch.__version__ : {torch.__version__}")
    print(f"torch.version.cuda : {cuda_compiled}")
    print(f"torch.cuda.is_available() : {cuda_ok}")

    ok, gpu_txt = _nvidia_smi_list()
    print("\n--- nvidia-smi -L ---")
    print(gpu_txt)

    if cuda_compiled and cuda_ok:
        print("\nCUDA-enabled PyTorch is active — GPU training will work.")
        return

    if cuda_compiled and not cuda_ok:
        print(
            "\nPyTorch includes CUDA libraries but cuda.is_available() is False.\n"
            "Try updating drivers, reinstalling CUDA-enabled PyTorch, or checking env vars "
            "(CUDA_VISIBLE_DEVICES)."
        )
        return

    if not INSTALL_CUDA_PYTORCH:
        print(
            "\nCPU-only PyTorch detected.\n"
            "To use RTX 4090:\n"
            "  1) Set INSTALL_CUDA_PYTORCH = True in this cell.\n"
            "  2) Run this cell (downloads CUDA 12.4 PyTorch).\n"
            "  3) Restart Jupyter kernel.\n"
            "  4) Set INSTALL_CUDA_PYTORCH = False and Run All.\n"
            "\nManual alternative (PowerShell / terminal):\n"
            "  pip uninstall -y torch torchvision torchaudio\n"
            "  pip install torch torchvision torchaudio "
            "--index-url https://download.pytorch.org/whl/cu124"
        )
        return

    print("\nInstalling PyTorch + torchvision + torchaudio with CUDA 12.4 ...")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "torch",
            "torchvision",
            "torchaudio",
            "--index-url",
            "https://download.pytorch.org/whl/cu124",
        ]
    )
    print(
        "\n>>> Done. Restart the Jupyter kernel now (Kernel → Restart).\n"
        ">>> Then set INSTALL_CUDA_PYTORCH = False and run all cells."
    )
    sys.exit(0)


main()


In [ ]:
import os
import json
import copy
import math
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pretty_midi
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── Reproducibility & CUDA/cuDNN ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

_use_cuda = torch.cuda.is_available()
_env_cudnn = os.environ.get("PYTORCH_DISABLE_CUDNN")
if _env_cudnn is None:
    DISABLE_CUDNN_FOR_STABILITY = os.name == "nt"
else:
    DISABLE_CUDNN_FOR_STABILITY = _env_cudnn.strip().lower() not in ("0", "false", "no")

if _use_cuda:
    torch.cuda.manual_seed_all(SEED)
    if DISABLE_CUDNN_FOR_STABILITY:
        torch.backends.cudnn.enabled = False
    else:
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
else:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150})

device = torch.device("cuda" if _use_cuda else "cpu")

if _use_cuda:
    torch.cuda.empty_cache()
    w = torch.randn(256, 256, device=device, dtype=torch.float32)
    _warm_out = w @ w
    torch.cuda.synchronize()
    del w, _warm_out

print(f"Using device: {device}")
if _use_cuda:
    idx = torch.cuda.current_device()
    print(f"  GPU [{idx}]: {torch.cuda.get_device_name(idx)}")
    print(f"  CUDA: {torch.version.cuda}  |  VRAM: {torch.cuda.get_device_properties(idx).total_memory / (1024**3):.1f} GB")
    print(f"  cuDNN enabled: {torch.backends.cudnn.enabled}")
else:
    print(f"  torch.version.cuda: {torch.version.cuda}")

# ── Output directories ──
os.makedirs("outputs/task4/generated_midis", exist_ok=True)
os.makedirs("outputs/plots", exist_ok=True)

print("Setup complete.")


### 1.1 — Model Architecture (re-implemented from Task 3)

We replicate the **exact same** `PositionalEncoding` and `MusicTransformer`
classes used in Task 3 so the notebook is fully self-contained and the
pretrained checkpoint loads without issues.

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (Vaswani et al., 2017)."""

    def __init__(self, d_model: int = 256, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding and apply dropout."""
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class MusicTransformer(nn.Module):
    """Decoder-only Transformer — matches Task 3 (checkpoint keys + forward).

    Same as Task 3: ``self.transformer`` (nn.TransformerEncoder), GELU FFN,
    genre conditioning after positional encoding, and PAD padding mask.
    """

    def __init__(self, vocab_size: int, d_model: int = 256, nhead: int = 8,
                 num_layers: int = 4, dim_feedforward: int = 1024,
                 dropout: float = 0.1, max_len: int = 512, num_genres: int = 5):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.genre_embedding = nn.Embedding(num_genres, d_model)
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=max_len, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_head = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.token_embedding.weight)
        nn.init.xavier_uniform_(self.genre_embedding.weight)
        nn.init.xavier_uniform_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def _generate_causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
        return mask.float().masked_fill(mask, float("-inf"))

    def forward(self, src: torch.Tensor, genre_ids: torch.Tensor | None = None) -> torch.Tensor:
        B, S = src.shape
        x = self.token_embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        if genre_ids is None:
            genre_ids = torch.zeros(B, dtype=torch.long, device=src.device)
        x = x + self.genre_embedding(genre_ids).unsqueeze(1)
        causal_mask = self._generate_causal_mask(S, src.device)
        pad_mask = src == PAD_TOKEN_ID
        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=pad_mask)
        return self.output_head(x)


print("Model classes defined.")

### 1.2 — Generation & Detokenization Utilities

In [ ]:
def generate(model, seed_tokens, max_len=512, temperature=0.8):
    """Autoregressively generate a token sequence via temperature sampling.

    Args:
        model: MusicTransformer instance.
        seed_tokens: 1-D list or tensor of seed token ids.
        max_len: Maximum total sequence length to produce.
        temperature: Softmax temperature (lower = more deterministic).

    Returns:
        List[int] — generated token ids (including seed).
    """
    model.eval()
    if isinstance(seed_tokens, list):
        tokens = seed_tokens[:]
    else:
        tokens = seed_tokens.tolist()

    with torch.no_grad():
        for _ in range(max_len - len(tokens)):
            inp = torch.tensor([tokens], dtype=torch.long, device=device)
            logits = model(inp)[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            tokens.append(next_token)
    return tokens


def detokenize(tokens, vocab, output_path=None):
    """Convert a token sequence back to a PrettyMIDI object.

    The vocabulary maps event strings (e.g. 'note_on_60', 'time_shift_50ms')
    to integer ids. This function reverses the mapping and reconstructs a
    MIDI file from the event stream.

    Args:
        tokens: List[int] — token ids.
        vocab: dict — token-string-to-id mapping.
        output_path: Optional path to write a .mid file.

    Returns:
        pretty_midi.PrettyMIDI object.
    """
    id_to_token = {v: k for k, v in vocab.items()}

    midi = pretty_midi.PrettyMIDI()
    instrument = pretty_midi.Instrument(program=0, name="Piano")

    current_time = 0.0
    active_notes = {}

    for tok_id in tokens:
        tok_str = id_to_token.get(tok_id, "")

        if tok_str.startswith("note_on_"):
            pitch = int(tok_str.split("_")[-1])
            active_notes[pitch] = current_time

        elif tok_str.startswith("note_off_"):
            pitch = int(tok_str.split("_")[-1])
            if pitch in active_notes:
                start = active_notes.pop(pitch)
                end = max(current_time, start + 0.05)
                note = pretty_midi.Note(
                    velocity=100, pitch=pitch,
                    start=start, end=end,
                )
                instrument.notes.append(note)

        elif tok_str.startswith("time_shift_"):
            ms = int(tok_str.split("_")[-1].replace("ms", ""))
            current_time += ms / 1000.0

        elif tok_str.startswith("velocity_"):
            pass  # velocity changes handled implicitly

    # Close any lingering notes
    for pitch, start in active_notes.items():
        instrument.notes.append(
            pretty_midi.Note(velocity=100, pitch=pitch,
                             start=start, end=start + 0.25)
        )

    midi.instruments.append(instrument)

    if output_path is not None:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        midi.write(output_path)

    return midi


print("generate() and detokenize() defined.")

### 1.3 — Load Vocabulary & Pretrained Model

In [ ]:
# ── Load vocabulary (same PAD handling as Task 3) ──
with open("outputs/processed/vocab.json", "r") as f:
    vocab = json.load(f)

VOCAB_SIZE = len(vocab)
if "PAD" not in vocab:
    PAD_TOKEN_ID = VOCAB_SIZE
    vocab["PAD"] = PAD_TOKEN_ID
    VOCAB_SIZE_WITH_PAD = VOCAB_SIZE + 1
else:
    PAD_TOKEN_ID = vocab["PAD"]
    VOCAB_SIZE_WITH_PAD = VOCAB_SIZE

vocab_size = VOCAB_SIZE_WITH_PAD
print(f"Base vocabulary size: {VOCAB_SIZE}")
print(f"Full vocabulary size (with PAD): {VOCAB_SIZE_WITH_PAD}")

# ── Instantiate model & load pretrained weights ──
pretrained_model = MusicTransformer(vocab_size=vocab_size).to(device)

checkpoint = torch.load("outputs/task3/best_model.pth", map_location=device, weights_only=False)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    pretrained_model.load_state_dict(checkpoint["model_state_dict"])
else:
    pretrained_model.load_state_dict(checkpoint)

pretrained_model.eval()
print("Pretrained Transformer loaded from outputs/task3/best_model.pth")

# ── Load token data (for seed sequences) ──
train_tokens = np.load("outputs/processed/train_tokens.npy", allow_pickle=True)
print(f"Training token sequences loaded — shape: {train_tokens.shape}")

### 1.4 — Note on Simulated Human Feedback

In a production RLHF pipeline the reward signal $r$ comes from **real human
annotators** who rate generated music on subjective quality dimensions
(melodic coherence, rhythmic stability, creativity, etc.). Training a
separate *reward model* on those annotations then provides a differentiable
proxy that can be queried at scale during RL training.

Because collecting human ratings is infeasible in this academic setting, we
design an **automated reward function** that combines four musically
meaningful metrics as a proxy for human preference. The same metrics are
later mapped to a simulated 1–5 Mean Opinion Score (MOS) to emulate a
listening survey.

---
## Section 2 — Reward Function Design

The reward combines four sub-scores, each normalized to **[0, 1]**:

| Weight | Sub-score | Intuition |
|:------:|-----------|----------|
| 0.30 | **Pitch Diversity** | Entropy of pitch-class histogram — varied melodies score higher |
| 0.30 | **Rhythmic Regularity** | Lower duration variance — stable rhythm scores higher |
| 0.20 | **Note Density** | ~5 notes/sec is the sweet spot; too sparse or dense is penalized |
| 0.20 | **Repetition Penalty** | Short-window pattern repetition is penalized |

In [ ]:
def compute_reward(midi_or_tokens, vocab):
    """Compute an automated reward score as a proxy for human preference.

    If *midi_or_tokens* is a list of ints it is first detokenized to a
    PrettyMIDI object (in memory only, no file written).

    Sub-scores (all in [0, 1]):
        - Pitch diversity:     entropy of 12-bin pitch-class histogram.
        - Rhythmic regularity: 1 / (1 + var(durations)).
        - Note density:        penalizes deviation from ~5 notes/sec.
        - Repetition penalty:  1 - ratio of repeated 16-token patterns.

    Returns:
        (total_reward, sub_scores_dict)
    """
    # ── Obtain PrettyMIDI object ──
    if isinstance(midi_or_tokens, list):
        midi = detokenize(midi_or_tokens, vocab, output_path=None)
        raw_tokens = midi_or_tokens
    elif isinstance(midi_or_tokens, pretty_midi.PrettyMIDI):
        midi = midi_or_tokens
        raw_tokens = []
    else:
        return 0.0, {"pitch_diversity": 0, "rhythmic_regularity": 0,
                     "note_density": 0, "repetition_penalty": 0}

    notes = [n for inst in midi.instruments for n in inst.notes]

    if len(notes) == 0:
        return 0.0, {"pitch_diversity": 0.0, "rhythmic_regularity": 0.0,
                     "note_density": 0.0, "repetition_penalty": 0.0}

    # ── (a) Pitch Diversity Score ──
    pitch_classes = [note.pitch % 12 for note in notes]
    hist = np.zeros(12)
    for pc in pitch_classes:
        hist[pc] += 1
    hist = hist / hist.sum()
    entropy = -sum(p * np.log(p) for p in hist if p > 0)
    pitch_diversity = entropy / np.log(12)

    # ── (b) Rhythmic Regularity Score ──
    durations = [note.end - note.start for note in notes]
    rhythmic_regularity = 1.0 / (1.0 + np.var(durations))

    # ── (c) Note Density Score ──
    total_time = max(n.end for n in notes) - min(n.start for n in notes)
    if total_time > 0:
        density = len(notes) / total_time
        note_density = float(np.clip(1.0 - abs(density - 5.0) / 5.0, 0.0, 1.0))
    else:
        note_density = 0.0

    # ── (d) Repetition Penalty ──
    if len(raw_tokens) > 16:
        window_size = 16
        patterns = []
        for i in range(len(raw_tokens) - window_size + 1):
            patterns.append(tuple(raw_tokens[i:i + window_size]))
        pattern_counts = Counter(patterns)
        repeated = sum(c - 1 for c in pattern_counts.values() if c > 1)
        repetition_ratio = repeated / max(len(patterns), 1)
        repetition_penalty = 1.0 - repetition_ratio
    else:
        repetition_penalty = 1.0

    # ── Total reward ──
    total_reward = (0.3 * pitch_diversity +
                    0.3 * rhythmic_regularity +
                    0.2 * note_density +
                    0.2 * repetition_penalty)

    sub_scores = {
        "pitch_diversity": round(float(pitch_diversity), 4),
        "rhythmic_regularity": round(float(rhythmic_regularity), 4),
        "note_density": round(float(note_density), 4),
        "repetition_penalty": round(float(repetition_penalty), 4),
    }
    return float(total_reward), sub_scores


print("compute_reward() defined.")

### 2.1 — Reward Distribution of Pretrained Model (50 Samples)

In [ ]:
NUM_PRETRAINED_SAMPLES = 50
pretrained_rewards = []

seed_seq = train_tokens[0][:16].tolist() if hasattr(train_tokens[0], 'tolist') else list(train_tokens[0][:16])

print(f"Generating {NUM_PRETRAINED_SAMPLES} samples from pretrained model...")
for i in tqdm(range(NUM_PRETRAINED_SAMPLES), desc="Pretrained sampling"):
    tokens = generate(pretrained_model, seed_seq, max_len=256, temperature=0.8)
    reward, _ = compute_reward(tokens, vocab)
    pretrained_rewards.append(reward)

pretrained_rewards = np.array(pretrained_rewards)
print(f"Pretrained model — mean reward: {pretrained_rewards.mean():.4f}, "
      f"std: {pretrained_rewards.std():.4f}")

# ── Histogram ──
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(pretrained_rewards, bins=15, edgecolor="black", alpha=0.75, color="steelblue")
ax.axvline(pretrained_rewards.mean(), color="red", linestyle="--",
           label=f"Mean = {pretrained_rewards.mean():.3f}")
ax.set_xlabel("Reward")
ax.set_ylabel("Frequency")
ax.set_title("Reward Distribution — Pretrained Model (50 samples)")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/plots/task4_pretrained_reward_dist.png", dpi=150)
plt.show()
print("Saved: outputs/plots/task4_pretrained_reward_dist.png")

---
## Section 3 — RLHF Training (Policy Gradient / REINFORCE)

We use the **REINFORCE** algorithm (Williams, 1992) to fine-tune the
pretrained Transformer. At each iteration the model generates a batch of
sequences, computes their rewards, and updates parameters to increase the
probability of high-reward sequences.

$$
\mathcal{L} = -\frac{1}{B} \sum_{i=1}^{B} \hat{r}_i \cdot \log p_\theta(X_i)
$$

where $\hat{r}_i$ is the baseline-subtracted, normalized reward.

**Resume:** `outputs/task4/rl_training_checkpoint.pth` is written after each iteration.


In [ ]:
def generate_with_log_probs(model, seed_tokens, max_len=256, temperature=0.8):
    """Generate a sequence while tracking per-token log-probabilities.

    Unlike the vanilla generate(), gradients are **not** detached so
    the computation graph is preserved for REINFORCE.

    Args:
        model: MusicTransformer in train mode.
        seed_tokens: 1-D list of seed token ids.
        max_len: Maximum total sequence length.
        temperature: Softmax temperature.

    Returns:
        tokens  : List[int] — full generated sequence.
        log_prob: torch.Tensor (scalar) — sum of log-probs of sampled tokens.
    """
    tokens = seed_tokens[:]
    log_probs = []

    for _ in range(max_len - len(tokens)):
        inp = torch.tensor([tokens], dtype=torch.long, device=device)
        logits = model(inp)[:, -1, :] / temperature
        log_softmax = F.log_softmax(logits, dim=-1)
        probs = log_softmax.exp()
        next_token = torch.multinomial(probs, num_samples=1)
        chosen_log_prob = log_softmax[0, next_token.item()]
        log_probs.append(chosen_log_prob)
        tokens.append(next_token.item())

    total_log_prob = torch.stack(log_probs).sum()
    return tokens, total_log_prob


print("generate_with_log_probs() defined.")

In [ ]:
# ── Create RL model (deep copy of pretrained) ──
rl_model = copy.deepcopy(pretrained_model).to(device)
rl_model.train()

optimizer = optim.Adam(rl_model.parameters(), lr=1e-5)

NUM_ITERATIONS = 20
BATCH_SIZE = 8
GEN_MAX_LEN = 256

RESUME_IF_AVAILABLE = True
CHECKPOINT_PATH = os.path.join("outputs", "task4", "rl_training_checkpoint.pth")

rl_log = []
start_iteration = 1

if RESUME_IF_AVAILABLE and os.path.isfile(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    rl_model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    rl_log = ckpt["rl_log"]
    completed = int(ckpt["iteration"])
    start_iteration = completed + 1
    print(
        f"Resumed RL checkpoint: iterations 1–{completed} done; "
        f"continuing from iteration {start_iteration}/{NUM_ITERATIONS}"
    )
else:
    print("Starting REINFORCE from iteration 1 (no checkpoint or RESUME_IF_AVAILABLE=False).")

print(f"Starting REINFORCE training — {NUM_ITERATIONS} iterations, "
      f"batch size {BATCH_SIZE}")
print("=" * 65)

if start_iteration > NUM_ITERATIONS:
    print(
        f"RL loop already finished for NUM_ITERATIONS={NUM_ITERATIONS}. "
        f"Raise NUM_ITERATIONS or delete {CHECKPOINT_PATH}."
    )
else:
    for iteration in range(start_iteration, NUM_ITERATIONS + 1):
        batch_rewards = []
        batch_log_probs = []

        for _ in range(BATCH_SIZE):
            tokens, log_prob = generate_with_log_probs(
                rl_model, seed_seq, max_len=GEN_MAX_LEN, temperature=0.8,
            )
            reward, _ = compute_reward(tokens, vocab)
            batch_rewards.append(reward)
            batch_log_probs.append(log_prob)

        rewards_t = torch.tensor(batch_rewards, dtype=torch.float32, device=device)
        r_mean = rewards_t.mean()
        r_std = rewards_t.std() + 1e-8
        normalized_rewards = (rewards_t - r_mean) / r_std

        log_probs_t = torch.stack(batch_log_probs)
        loss = -(normalized_rewards.detach() * log_probs_t).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        mean_r = float(r_mean)
        std_r = float(rewards_t.std())
        mean_lp = float(log_probs_t.mean().detach())

        rl_log.append({
            "iteration": iteration,
            "mean_reward": round(mean_r, 4),
            "std_reward": round(std_r, 4),
            "mean_log_prob": round(mean_lp, 4),
            "loss": round(float(loss.item()), 4),
        })

        torch.save(
            {
                "iteration": iteration,
                "model_state_dict": rl_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "rl_log": rl_log,
                "num_iterations": NUM_ITERATIONS,
            },
            CHECKPOINT_PATH,
        )

        print(f"[Iter {iteration:>2}/{NUM_ITERATIONS}]  "
              f"mean_reward={mean_r:.4f}  std={std_r:.4f}  "
              f"mean_log_prob={mean_lp:.2f}  loss={loss.item():.4f}")

    print("=" * 65)
    print("REINFORCE training complete.")

rl_df = pd.DataFrame(rl_log)
rl_df.to_csv("outputs/task4/rl_training_log.csv", index=False)
print("Training log saved to outputs/task4/rl_training_log.csv")


---
## Section 4 — Visualizations

### 4.1 — Mean Reward per RL Iteration (Learning Curve)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
iters = rl_df["iteration"].values
means = rl_df["mean_reward"].values
stds = rl_df["std_reward"].values

ax.plot(iters, means, marker="o", color="teal", linewidth=2, label="Mean Reward")
ax.fill_between(iters, means - stds, means + stds, alpha=0.2, color="teal")
ax.errorbar(iters, means, yerr=stds, fmt="none", ecolor="gray", capsize=3, alpha=0.5)

ax.set_xlabel("RL Iteration")
ax.set_ylabel("Reward")
ax.set_title("REINFORCE Learning Curve — Mean Reward per Iteration")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/plots/task4_rl_reward_curve.png", dpi=150)
plt.show()
print("Saved: outputs/plots/task4_rl_reward_curve.png")

### 4.2 — Reward Distribution: Pre-RL vs Post-RL

In [ ]:
NUM_COMPARISON_SAMPLES = 50

print("Generating 50 post-RL samples for comparison...")
post_rl_rewards = []
rl_model.eval()
for i in tqdm(range(NUM_COMPARISON_SAMPLES), desc="Post-RL sampling"):
    tokens = generate(rl_model, seed_seq, max_len=256, temperature=0.8)
    reward, _ = compute_reward(tokens, vocab)
    post_rl_rewards.append(reward)

post_rl_rewards = np.array(post_rl_rewards)
print(f"Post-RL — mean reward: {post_rl_rewards.mean():.4f}, "
      f"std: {post_rl_rewards.std():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].hist(pretrained_rewards, bins=15, edgecolor="black", alpha=0.75,
             color="steelblue", label="Pre-RL")
axes[0].axvline(pretrained_rewards.mean(), color="red", linestyle="--",
                label=f"Mean = {pretrained_rewards.mean():.3f}")
axes[0].set_xlabel("Reward")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Pre-RL Reward Distribution")
axes[0].legend()

axes[1].hist(post_rl_rewards, bins=15, edgecolor="black", alpha=0.75,
             color="coral", label="Post-RL")
axes[1].axvline(post_rl_rewards.mean(), color="red", linestyle="--",
                label=f"Mean = {post_rl_rewards.mean():.3f}")
axes[1].set_xlabel("Reward")
axes[1].set_title("Post-RL Reward Distribution")
axes[1].legend()

fig.suptitle("Reward Comparison — Before vs After RLHF Fine-Tuning", fontsize=14)
plt.tight_layout()
plt.savefig("outputs/plots/task4_reward_comparison.png", dpi=150)
plt.show()
print("Saved: outputs/plots/task4_reward_comparison.png")

### 4.3 — Piano Roll Comparison: Pre-RL vs Post-RL

In [ ]:
print("Generating piano-roll comparison samples...")

pre_tokens = generate(pretrained_model, seed_seq, max_len=256, temperature=0.8)
pre_midi = detokenize(pre_tokens, vocab)

rl_model.eval()
post_tokens = generate(rl_model, seed_seq, max_len=256, temperature=0.8)
post_midi = detokenize(post_tokens, vocab)

pre_roll = pre_midi.get_piano_roll(fs=50)
post_roll = post_midi.get_piano_roll(fs=50)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].imshow(pre_roll, aspect="auto", origin="lower", cmap="magma",
               interpolation="nearest")
axes[0].set_title("Piano Roll — Pre-RL (Pretrained Model)")
axes[0].set_ylabel("MIDI Pitch")
axes[0].set_xlabel("Time Frame (50 fps)")

axes[1].imshow(post_roll, aspect="auto", origin="lower", cmap="magma",
               interpolation="nearest")
axes[1].set_title("Piano Roll — Post-RL (RLHF-Tuned Model)")
axes[1].set_ylabel("MIDI Pitch")
axes[1].set_xlabel("Time Frame (50 fps)")

plt.tight_layout()
plt.savefig("outputs/plots/task4_piano_roll_comparison.png", dpi=150)
plt.show()
print("Saved: outputs/plots/task4_piano_roll_comparison.png")

---
## Section 5 — Human Listening Survey Simulation

We simulate a Mean-Opinion-Score (MOS) listening test by mapping the
automated reward to a 1–5 integer scale:

$$
\text{human\_score} = 1 + 4 \times r
$$

In [ ]:
NUM_SURVEY = 10
survey_rows = []

print(f"Generating {NUM_SURVEY} pre-RL and {NUM_SURVEY} post-RL samples for survey...")

for idx in tqdm(range(NUM_SURVEY), desc="Pre-RL survey"):
    tokens = generate(pretrained_model, seed_seq, max_len=256, temperature=0.8)
    reward, subs = compute_reward(tokens, vocab)
    human_score = 1 + 4 * reward
    survey_rows.append({
        "sample_id": idx + 1,
        "model": "pre_rl",
        "pitch_diversity": subs["pitch_diversity"],
        "rhythm_score": subs["rhythmic_regularity"],
        "note_density": subs["note_density"],
        "rep_penalty": subs["repetition_penalty"],
        "total_reward": round(reward, 4),
        "human_score_simulated": round(human_score, 2),
    })

rl_model.eval()
for idx in tqdm(range(NUM_SURVEY), desc="Post-RL survey"):
    tokens = generate(rl_model, seed_seq, max_len=256, temperature=0.8)
    reward, subs = compute_reward(tokens, vocab)
    human_score = 1 + 4 * reward
    survey_rows.append({
        "sample_id": idx + 1,
        "model": "post_rl",
        "pitch_diversity": subs["pitch_diversity"],
        "rhythm_score": subs["rhythmic_regularity"],
        "note_density": subs["note_density"],
        "rep_penalty": subs["repetition_penalty"],
        "total_reward": round(reward, 4),
        "human_score_simulated": round(human_score, 2),
    })

survey_df = pd.DataFrame(survey_rows)
survey_df.to_csv("outputs/task4/survey_results.csv", index=False)
print("Survey results saved to outputs/task4/survey_results.csv")
print()
print(survey_df.groupby("model")[["total_reward", "human_score_simulated"]].mean())

In [ ]:
# ── Grouped bar chart of simulated human scores ──
mean_scores = survey_df.groupby("model")["human_score_simulated"].mean()
std_scores = survey_df.groupby("model")["human_score_simulated"].std()

fig, ax = plt.subplots(figsize=(7, 5))
models = ["pre_rl", "post_rl"]
colors = ["steelblue", "coral"]
bars = ax.bar(models, [mean_scores[m] for m in models],
              yerr=[std_scores[m] for m in models],
              color=colors, edgecolor="black", capsize=6, alpha=0.85)

for bar, m in zip(bars, models):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.08,
            f"{mean_scores[m]:.2f}", ha="center", va="bottom", fontweight="bold")

ax.set_ylabel("Simulated Human Score (1-5)")
ax.set_xlabel("Model")
ax.set_title("Simulated Human Listening Scores — Pre-RL vs Post-RL")
ax.set_ylim(0, 5.5)
ax.set_xticklabels(["Pre-RL", "Post-RL"])
plt.tight_layout()
plt.savefig("outputs/plots/task4_human_scores.png", dpi=150)
plt.show()
print("Saved: outputs/plots/task4_human_scores.png")

### 5.1 — Human Listening Survey Methodology (Template)

In a real deployment, we would conduct a listening survey with **10+
participants** following this protocol:

**Methodology:**
1. Recruit 10–20 participants with varying musical backgrounds.
2. Each participant listens to 20 randomly ordered 30-second clips (10
   pre-RL, 10 post-RL) without knowing which model produced each clip.
3. After each clip, the participant rates it on a **1–5 MOS scale**:

| Score | Label | Description |
|:-----:|-------|-------------|
| 5 | Excellent | Musically compelling, coherent structure |
| 4 | Good | Pleasant, mostly coherent |
| 3 | Fair | Acceptable but noticeable artifacts |
| 2 | Poor | Significant quality issues |
| 1 | Bad | Unpleasant, incoherent |

**Scoring Rubric Dimensions** (each 1–5):
- **Melodic Quality**: Are the pitch sequences musically meaningful?
- **Rhythmic Stability**: Is the timing regular and natural?
- **Creativity / Variety**: Does the piece avoid excessive repetition?
- **Overall Enjoyment**: Would you listen to this voluntarily?

**Statistical Analysis:**
- Compute mean and standard deviation per model.
- Perform a **paired t-test** (or Wilcoxon signed-rank test if normality
  is violated) to determine whether the difference is statistically
  significant at $\alpha = 0.05$.
- Report Cohen's $d$ effect size.

---
## Section 6 — Final Generated Outputs

In [ ]:
NUM_FINAL = 10
os.makedirs("outputs/task4/generated_midis", exist_ok=True)

print(f"Generating {NUM_FINAL} MIDI compositions with the RL-tuned model...")
rl_model.eval()

for i in tqdm(range(1, NUM_FINAL + 1), desc="Generating MIDIs"):
    tokens = generate(rl_model, seed_seq, max_len=512, temperature=0.8)
    out_path = f"outputs/task4/generated_midis/generated_{i}.mid"
    detokenize(tokens, vocab, output_path=out_path)
    print(f"  Saved: {out_path}")

print(f"\nAll {NUM_FINAL} compositions saved to outputs/task4/generated_midis/")

In [ ]:
# ── Save RL-tuned model checkpoint ──
torch.save({
    "model_state_dict": rl_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "rl_training_log": rl_log,
    "vocab_size": vocab_size,
}, "outputs/task4/rl_tuned_model.pth")

print("RL-tuned model checkpoint saved to outputs/task4/rl_tuned_model.pth")

---
## Section 7 — Before vs After Analysis

### 7.1 — Compute Pre-RL and Post-RL Aggregate Metrics

In [ ]:
# ── Load Task 3 baseline metrics ──
task3_metrics_path = "outputs/task3/metrics.csv"
if os.path.exists(task3_metrics_path):
    task3_df = pd.read_csv(task3_metrics_path)
    print("Loaded Task 3 metrics from", task3_metrics_path)
    print(task3_df.tail())
else:
    print(f"Warning: {task3_metrics_path} not found — using pretrained reward samples instead.")
    task3_df = None

# ── Aggregate sub-scores from the survey data ──
pre_rl_survey = survey_df[survey_df["model"] == "pre_rl"]
post_rl_survey = survey_df[survey_df["model"] == "post_rl"]

dimensions = ["pitch_diversity", "rhythm_score", "note_density",
              "rep_penalty", "human_score_simulated"]
display_names = ["Pitch Diversity", "Rhythm Score", "Note Density",
                 "Repetition Penalty", "Human Score"]

pre_means = [pre_rl_survey[d].mean() for d in dimensions]
post_means = [post_rl_survey[d].mean() for d in dimensions]

# Normalize human score to [0, 1] for radar chart
pre_means_norm = pre_means[:4] + [pre_means[4] / 5.0]
post_means_norm = post_means[:4] + [post_means[4] / 5.0]

comparison_df = pd.DataFrame({
    "Dimension": display_names,
    "Pre-RL Mean": [round(v, 4) for v in pre_means],
    "Post-RL Mean": [round(v, 4) for v in post_means],
    "Change": [round(post - pre, 4) for pre, post in zip(pre_means, post_means)],
})

print("\n===== Before vs After RLHF Comparison =====")
print(comparison_df.to_string(index=False))

comparison_df.to_csv("outputs/task4/before_after_metrics.csv", index=False)
print("\nSaved: outputs/task4/before_after_metrics.csv")

### 7.2 — Radar Chart: Pre-RL vs Post-RL

In [ ]:
labels = display_names
num_vars = len(labels)

angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

pre_values = pre_means_norm + pre_means_norm[:1]
post_values = post_means_norm + post_means_norm[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

ax.plot(angles, pre_values, "o-", linewidth=2, color="steelblue", label="Pre-RL")
ax.fill(angles, pre_values, alpha=0.15, color="steelblue")

ax.plot(angles, post_values, "o-", linewidth=2, color="coral", label="Post-RL")
ax.fill(angles, post_values, alpha=0.15, color="coral")

ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title("Radar Chart — Pre-RL vs Post-RL Quality Dimensions",
             fontsize=14, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/plots/task4_radar_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/plots/task4_radar_chart.png")

In [ ]:
print("\n" + "=" * 65)
print("  Task 4 — RLHF Fine-Tuning Complete")
print("=" * 65)
print()
print("Outputs produced:")
print("  - outputs/task4/rl_tuned_model.pth")
print("  - outputs/task4/rl_training_log.csv")
print("  - outputs/task4/survey_results.csv")
print("  - outputs/task4/before_after_metrics.csv")
print("  - outputs/task4/generated_midis/generated_1.mid .. generated_10.mid")
print()
print("Plots produced:")
print("  - outputs/plots/task4_pretrained_reward_dist.png")
print("  - outputs/plots/task4_rl_reward_curve.png")
print("  - outputs/plots/task4_reward_comparison.png")
print("  - outputs/plots/task4_piano_roll_comparison.png")
print("  - outputs/plots/task4_human_scores.png")
print("  - outputs/plots/task4_radar_chart.png")